# Pretrained Inference Demo

This notebook inspects the prediction files produced by the trained validity heads. This is the easiest beginner path for inference: each row is a candidate plan with a predicted probability of validity.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

DATASET = ROOT / "data" / "validity_dataset"
CANDIDATES = DATASET / "candidates"
FEATURES = DATASET / "features"
HEADS = ROOT / "models" / "validity_heads"
RESULTS = ROOT / "results" / "analysis"

print(f"Repository root: {ROOT}")

Repository root: C:\Users\visha\OneDrive\Desktop\tokenizer-paper-workspace\planfm-validity


In [2]:
family = "dd_xgb_wl_delta"
source_seed = 13
head_seed = 13
pred_path = HEADS / family / f"source_seed_{source_seed}" / f"head_seed_{head_seed}" / "predictions.csv"
preds = pd.read_csv(pred_path)
print(f"Loaded: {pred_path.relative_to(ROOT)}")
preds.head()

Loaded: models\validity_heads\dd_xgb_wl_delta\source_seed_13\head_seed_13\predictions.csv


,split,candidate_id,domain,problem,corruption_type,label_valid,prob_valid,pred_valid
0,train,blocks::train::probBLOCKS-4-0::000::gold,blocks,probBLOCKS-4-0,gold,1,0.801297,1
1,train,blocks::train::probBLOCKS-4-0::001::truncate,blocks,probBLOCKS-4-0,truncate,0,0.043198,0
2,train,blocks::train::probBLOCKS-4-0::002::delete,blocks,probBLOCKS-4-0,delete,0,0.023576,0
3,train,blocks::train::probBLOCKS-4-0::003::swap,blocks,probBLOCKS-4-0,swap,0,0.031631,0
4,train,blocks::train::probBLOCKS-4-0::004::replace,blocks,probBLOCKS-4-0,replace,0,0.024512,0


In [3]:
from sklearn.metrics import confusion_matrix, classification_report

for split, group in preds.groupby("split"):
    y_true = group["label_valid"].to_numpy()
    y_pred = group["pred_valid"].to_numpy()
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    print(f"{split:20s}  n={len(group):4d}  tn={tn:4d} fp={fp:3d} fn={fn:3d} tp={tp:3d}")

test-extrapolation    n=1365  tn=1090 fp=  2 fn=242 tp= 31
test-interpolation    n= 260  tn= 191 fp= 17 fn=  5 tp= 47
train                 n=1156  tn= 909 fp= 16 fn=  6 tp=225
validation            n= 165  tn= 130 fp=  2 fn=  3 tp= 30


In [4]:
# Join predictions back to candidate plans so I can inspect the action sequence.
def load_candidates(split):
    return pd.read_json(CANDIDATES / f"{split}.jsonl", lines=True)

candidate_frames = [load_candidates(split) for split in preds["split"].unique()]
candidates = pd.concat(candidate_frames, ignore_index=True)
joined = preds.merge(candidates[["candidate_id", "plan", "plan_len", "gold_plan_len", "label_executable"]], on="candidate_id", how="left")
joined.head()

,split,candidate_id,domain,problem,corruption_type,label_valid,prob_valid,pred_valid,plan,plan_len,gold_plan_len,label_executable
0,train,blocks::train::probBLOCKS-4-0::000::gold,blocks,probBLOCKS-4-0,gold,1,0.801297,1,"[(pick-up b), (stack b a), (pick-up c), (stack...",6,6,1
1,train,blocks::train::probBLOCKS-4-0::001::truncate,blocks,probBLOCKS-4-0,truncate,0,0.043198,0,[],0,6,0
2,train,blocks::train::probBLOCKS-4-0::002::delete,blocks,probBLOCKS-4-0,delete,0,0.023576,0,"[(pick-up b), (stack b a), (stack c b), (pick-...",5,6,0
3,train,blocks::train::probBLOCKS-4-0::003::swap,blocks,probBLOCKS-4-0,swap,0,0.031631,0,"[(pick-up b), (stack b a), (pick-up c), (stack...",6,6,0
4,train,blocks::train::probBLOCKS-4-0::004::replace,blocks,probBLOCKS-4-0,replace,0,0.024512,0,"[(pick-up b), (stack b a), (pick-up c), (stack...",6,6,0


In [5]:
def show_prediction(candidate_id):
    row = joined.loc[joined["candidate_id"] == candidate_id].iloc[0]
    print(f"candidate_id: {row.candidate_id}")
    print(f"split/domain/problem: {row.split} / {row.domain} / {row.problem}")
    print(f"corruption_type: {row.corruption_type}")
    print(f"label_valid: {row.label_valid}; pred_valid: {row.pred_valid}; prob_valid: {row.prob_valid:.3f}")
    print(f"label_executable: {row.label_executable}; plan_len: {row.plan_len}; gold_plan_len: {row.gold_plan_len}")
    print("plan:")
    for step, action in enumerate(row.plan, start=1):
        print(f"  {step:02d}. {action}")

show_prediction(joined.iloc[0].candidate_id)

candidate_id: blocks::train::probBLOCKS-4-0::000::gold
split/domain/problem: train / blocks / probBLOCKS-4-0
corruption_type: gold
label_valid: 1; pred_valid: 1; prob_valid: 0.801
label_executable: 1; plan_len: 6; gold_plan_len: 6
plan:
  01. (pick-up b)
  02. (stack b a)
  03. (pick-up c)
  04. (stack c b)
  05. (pick-up d)
  06. (stack d c)


In [6]:
# A high-confidence invalid example.
invalid = joined.query("label_valid == 0").sort_values("prob_valid").iloc[0]
show_prediction(invalid.candidate_id)

candidate_id: visitall-from-everywhere::train::w07h02-01::004::replace
split/domain/problem: train / visitall-from-everywhere / w07h02-01
corruption_type: replace
label_valid: 0; pred_valid: 0; prob_valid: 0.022
label_executable: 0; plan_len: 13; gold_plan_len: 13
plan:
  01. (move loc-x5-y1 loc-x6-y1)
  02. (move loc-x6-y1 loc-x6-y0)
  03. (move loc-x6-y0 loc-x5-y0)
  04. (move loc-x5-y0 loc-x4-y0)
  05. (move loc-x4-y0 loc-x4-y1)
  06. (move loc-x4-y1 loc-x3-y1)
  07. (move loc-x1-y1 loc-x0-y1)
  08. (move loc-x3-y0 loc-x2-y0)
  09. (move loc-x2-y0 loc-x2-y1)
  10. (move loc-x2-y1 loc-x1-y1)
  11. (move loc-x1-y1 loc-x0-y1)
  12. (move loc-x0-y1 loc-x0-y0)
  13. (move loc-x0-y0 loc-x1-y0)


In [7]:
# A valid example from the extrapolation split.
valid_extra = joined.query("split == 'test-extrapolation' and label_valid == 1").sort_values("prob_valid", ascending=False).iloc[0]
show_prediction(valid_extra.candidate_id)

candidate_id: visitall-from-everywhere::test-extrapolation::w03h07-01::000::gold
split/domain/problem: test-extrapolation / visitall-from-everywhere / w03h07-01
corruption_type: gold
label_valid: 1; pred_valid: 1; prob_valid: 0.942
label_executable: 1; plan_len: 20; gold_plan_len: 20
plan:
  01. (move loc-x0-y4 loc-x0-y5)
  02. (move loc-x0-y5 loc-x0-y6)
  03. (move loc-x0-y6 loc-x1-y6)
  04. (move loc-x1-y6 loc-x2-y6)
  05. (move loc-x2-y6 loc-x2-y5)
  06. (move loc-x2-y5 loc-x1-y5)
  07. (move loc-x1-y5 loc-x1-y4)
  08. (move loc-x1-y4 loc-x2-y4)
  09. (move loc-x2-y4 loc-x2-y3)
  10. (move loc-x2-y3 loc-x1-y3)
  11. (move loc-x1-y3 loc-x0-y3)
  12. (move loc-x0-y3 loc-x0-y2)
  13. (move loc-x0-y2 loc-x0-y1)
  14. (move loc-x0-y1 loc-x0-y0)
  15. (move loc-x0-y0 loc-x1-y0)
  16. (move loc-x1-y0 loc-x2-y0)
  17. (move loc-x2-y0 loc-x2-y1)
  18. (move loc-x2-y1 loc-x1-y1)
  19. (move loc-x1-y1 loc-x1-y2)
  20. (move loc-x1-y2 loc-x2-y2)
